In [ ]:
from phisolve.problems.miqp import MIQP
from phisolve.backends.commons import BackendParams
from phisolve.solvers.phi_miqp import PhiMIQPParams, PhiMIQP
import scipy as sp
import numpy as np

n = 100
density = 0.1
seed = 42
rng = np.random.default_rng(seed=seed)
def rvs(size=None, random_state=rng):
    return random_state.standard_normal(size)
Q = sp.sparse.random(n, n, density, random_state=rng, data_rvs=rvs)
Q = Q + Q.T
w = np.random.uniform(-1, 1, (n))

problem = MIQP(Q, w, n_binary_vars=len(w))

In [ ]:
n_shots = 100
n_steps = 15000
device = "cpu"

In [ ]:
import jax
from phisolve.utils.jax_utils import jax_device
jax.config.update("jax_platforms", jax_device(device))

In [ ]:
from phisolve.refiners.jax_adam import JaxAdam

refine = JaxAdam(device=device).refine
backend_params = BackendParams(n_shots=n_shots, n_steps=n_steps, ballistic=False, slow_a=True, seed=seed, device=device)
base_solver_params = PhiMIQPParams(refine=refine, backend_params=backend_params)
solver = PhiMIQP(problem)
res = solver.run(base_solver_params)
print(res.succ_prob())
print(res.detailed_time)
print(res.minimum)